# Alberta Electoral Boundary Audit — Interactive Explorer

**Phase 4C · Official Elections Alberta shapefiles · May 2026**

This notebook reproduces the key statistical findings from the audit without requiring a local checkout of the full repository (no large shapefiles, no MCMC re-run). It fetches the pre-computed output files directly from GitHub.

**Reproducibility note:** To pin to a specific commit, replace `main` in all URLs below with a commit SHA (e.g. `d8f101a`). The canonical outputs committed to `main` were generated by `packing_cracking_analysis.py` v0.3 and `mcmc_ensemble_canonical.py` with seed `1432864451` (1,010,000 plans across 4 chains).

---
Repository: https://github.com/Ixby/alberta-electoral-boundaries-audit  
Academic report: `reports/academic/report_academic.md`  
Pre-registration: OSF:6pt83, AsPredicted:#289,469, AsPredicted:#289,451

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import json, io, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

BASE = 'https://raw.githubusercontent.com/Ixby/alberta-electoral-boundaries-audit/main'

def fetch_json(path):
    r = requests.get(f'{BASE}/{path}', timeout=30)
    r.raise_for_status()
    return r.json()

def fetch_csv(path):
    r = requests.get(f'{BASE}/{path}', timeout=30)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))

print('Libraries loaded.')

In [ ]:
# ── Load pre-computed outputs ─────────────────────────────────────────────────
phase4c   = fetch_json('data/outputs/phase4c_canonical_results.json')
real_map  = fetch_json('data/outputs/simulation_real_map_scores_canonical.json')
conv_diag = fetch_json('data/outputs/simulation_convergence_diagnostics_canonical.json')
ensemble  = fetch_csv('data/outputs/simulated_ensemble_percentiles_canonical.csv')

print('Files fetched.')
print(f'Ensemble rows: {len(ensemble):,}')
print(f'Ensemble columns: {list(ensemble.columns)}')

---
## 1. Key Findings Table

All numbers sourced directly from `phase4c_canonical_results.json` (S-M sign convention: positive EG = UCP structural advantage).

In [ ]:
maj = phase4c['majority']
mn  = phase4c['minority']

total_partisan = maj['total_ndp'] + maj['total_ucp']

maj_eg_pct    = maj['eg'] * 100
mn_eg_pct     = mn['eg']  * 100
maj_wasted    = round(maj['eg'] * total_partisan)
mn_wasted     = round(mn['eg']  * total_partisan)

maj_ndp_seats = round(maj['seats_at_50'] * maj['n_eds'])
mn_ndp_seats  = round(mn['seats_at_50']  * mn['n_eds'])

summary = pd.DataFrame([
    ['Total partisan votes (NDP + UCP)',
     f'{total_partisan:,}', f'{total_partisan:,}'],
    ['Structural vote-to-seat imbalance (efficiency gap)',
     f'+{maj_eg_pct:.2f}%', f'+{mn_eg_pct:.2f}%'],
    ['Excess wasted NDP votes (imbalance × total votes)',
     f'~{maj_wasted:,}', f'~{mn_wasted:,}'],
    ['NDP seats at a perfectly tied (50/50) provincial vote',
     f'{maj_ndp_seats} / 89', f'{mn_ndp_seats} / 89'],
    ['Seat gap (minority minus majority at 50/50)',
     '—', f'{mn_ndp_seats - maj_ndp_seats:+d} NDP seats'],
    ['Mean-median difference',
     f"{maj['mean_median']*100:+.2f} pp", f"{mn['mean_median']*100:+.2f} pp"],
    ['Declination',
     f"{maj['declination']:+.4f}", f"{mn['declination']:+.4f}"],
], columns=['Metric', 'Majority map', 'Minority map'])

summary.set_index('Metric', inplace=True)
summary

---
## 2. Efficiency Gap — Maps vs. Neutral Ensemble

The histogram shows the distribution of efficiency-gap values across all plans in the 1,010,000-map neutral ensemble. Vertical lines mark where each commission map falls.

Positive values = UCP structural advantage. The audit's outlier threshold is the 95th percentile of the ensemble.

In [ ]:
# Identify the EG column (may vary by version)
eg_col = next((c for c in ensemble.columns if 'efficiency_gap' in c.lower() or c.lower() == 'eg'), None)
if eg_col is None:
    print('Available columns:', list(ensemble.columns))
    raise KeyError('No efficiency-gap column found — check column names above')

eg_vals = ensemble[eg_col].dropna() * 100  # convert to percent

p5  = np.percentile(eg_vals, 5)
p95 = np.percentile(eg_vals, 95)
p_maj = (eg_vals < maj_eg_pct).mean() * 100
p_mn  = (eg_vals < mn_eg_pct).mean() * 100

fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(eg_vals, bins=120, color='#aec6cf', edgecolor='none', alpha=0.85,
        label=f'Neutral ensemble ({len(eg_vals):,} plans)')

ax.axvline(p95, color='#888', lw=1, ls='--', label=f'95th pct outlier line ({p95:.1f}%)')
ax.axvline(maj_eg_pct, color='#1d6a27', lw=2.2,
           label=f'Majority map: +{maj_eg_pct:.2f}% (p{p_maj:.0f})')
ax.axvline(mn_eg_pct,  color='#922b21', lw=2.2,
           label=f'Minority map: +{mn_eg_pct:.2f}% (p{p_mn:.0f})')

ax.set_xlabel('Efficiency gap (positive = UCP structural advantage, %)')
ax.set_ylabel('Number of neutral plans')
ax.set_title('Efficiency gap: commission maps vs. 1,010,000-plan neutral ensemble')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'Majority map: p{p_maj:.1f} — inside normal range (below p95 outlier threshold)')
print(f'Minority map: p{p_mn:.1f} — above the p95 outlier threshold')

---
## 3. Seats@50/50 — Maps vs. Neutral Ensemble

Holds the provincial vote at exactly 50/50 (NDP/UCP) and counts how many seats each map allocates to the UCP. A neutral Alberta map produces a median near 44.8% UCP seats (NDP has a slight efficiency advantage from urban concentration).

In [ ]:
s50_col = next((c for c in ensemble.columns
                if 'seat' in c.lower() and '50' in c), None)
if s50_col is None:
    print('Available columns:', list(ensemble.columns))
    raise KeyError('No seats@50 column found')

s50_vals = ensemble[s50_col].dropna() * 100  # UCP seat fraction

# Real map UCP seat fractions at 50/50 (= 1 - NDP fraction)
maj_ucp_s50 = (1 - maj['seats_at_50']) * 100
mn_ucp_s50  = (1 - mn['seats_at_50'])  * 100

p_maj_s50 = (s50_vals < maj_ucp_s50).mean() * 100
p_mn_s50  = (s50_vals < mn_ucp_s50).mean()  * 100

fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(s50_vals, bins=100, color='#c5d8e8', edgecolor='none', alpha=0.85,
        label=f'Neutral ensemble ({len(s50_vals):,} plans)')
ax.axvline(np.percentile(s50_vals, 95), color='#888', lw=1, ls='--',
           label=f'95th pct outlier line')
ax.axvline(maj_ucp_s50, color='#1d6a27', lw=2.2,
           label=f'Majority map: {maj_ucp_s50:.1f}% UCP seats (p{p_maj_s50:.0f})')
ax.axvline(mn_ucp_s50,  color='#922b21', lw=2.2,
           label=f'Minority map: {mn_ucp_s50:.1f}% UCP seats (p{p_mn_s50:.1f})')

ax.set_xlabel('UCP seat share at a 50/50 tied provincial vote (%)')
ax.set_ylabel('Number of neutral plans')
ax.set_title('Seats at 50/50 tied vote: commission maps vs. 1,010,000-plan neutral ensemble')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

n_eds = maj['n_eds']
print(f'Majority map: {maj_ucp_s50:.1f}% UCP seats = {round(maj_ucp_s50/100*n_eds)} UCP / '
      f'{round(maj["seats_at_50"]*n_eds)} NDP at a tied vote (p{p_maj_s50:.0f})')
print(f'Minority map: {mn_ucp_s50:.1f}% UCP seats = {round(mn_ucp_s50/100*n_eds)} UCP / '
      f'{round(mn["seats_at_50"]*n_eds)} NDP at a tied vote (p{p_mn_s50:.1f})')

---
## 4. Metric Summary — All Four Measures Side by Side

In [ ]:
metrics = ['efficiency_gap', 'mean_median', 'declination', 'seats_at_50_50']
labels  = ['Efficiency gap', 'Mean-median', 'Declination', 'Seats@50/50 (UCP)']

# Real-map values from simulation_real_map_scores_canonical.json
real_maj = real_map.get('majority_2026', {})
real_mn  = real_map.get('minority_2026', {})

rows = []
for m, lbl in zip(metrics, labels):
    mv = real_maj.get(m)
    nv = real_mn.get(m)
    # Ensemble percentile if column exists
    col = next((c for c in ensemble.columns if m.lower().replace('_','') in c.lower().replace('_','')), None)
    if col is not None and mv is not None:
        pct_maj = (ensemble[col].dropna() < mv).mean() * 100
        pct_mn  = (ensemble[col].dropna() < nv).mean() * 100 if nv is not None else None
    else:
        pct_maj = pct_mn = None
    rows.append({
        'Metric': lbl,
        'Majority (raw)': f'{mv:.4f}' if mv is not None else 'n/a',
        'Majority (pctile)': f'p{pct_maj:.0f}' if pct_maj is not None else '—',
        'Minority (raw)': f'{nv:.4f}' if nv is not None else 'n/a',
        'Minority (pctile)': f'p{pct_mn:.0f}' if pct_mn is not None else '—',
    })

df_metrics = pd.DataFrame(rows).set_index('Metric')
df_metrics

---
## 5. MCMC Convergence Diagnostics

The ensemble is only valid if the Markov chains converged. Gelman-Rubin R̂ < 1.05 is the standard acceptance threshold; < 1.02 is excellent.

In [ ]:
print('=== MCMC Convergence Diagnostics ===')
print(f"Source: simulation_convergence_diagnostics_canonical.json\n")

if isinstance(conv_diag, dict):
    for k, v in conv_diag.items():
        if isinstance(v, dict):
            print(f'  {k}:')
            for sub_k, sub_v in v.items():
                print(f'    {sub_k}: {sub_v}')
        else:
            print(f'  {k}: {v}')
elif isinstance(conv_diag, list):
    for row in conv_diag[:20]:
        print(row)

---
## 6. What This Does Not Show

This notebook reproduces the pre-computed outputs only. It does not:

- Re-run the MCMC ensemble (1,010,000 plans, ~6 hours on a 4-core laptop)
- Re-run the Phase 4C spatial vote attribution (requires the GeoPackage shapefiles)
- Reproduce the SZAT (swing-zone) or Mahalanobis joint tests

To fully reproduce from raw data, clone the repository and follow `docs/REPRODUCING.md`.

---

## Disclosure

The author is a student at Mount Royal University. This research was conducted independently and was not commissioned as coursework. The author has in the past donated to and volunteered for the NDP; this is disclosed because it represents a potential source of motivated reasoning. The structural safeguard is symmetric methodology: every test was applied identically to both maps, and all tests were pre-registered before results were examined. This research received no external funding.

Pre-registration: **OSF:6pt83**, **AsPredicted:#289,469**, **AsPredicted:#289,451**  
Full academic report: `reports/academic/report_academic.md`